## 데이터 수집 자동화

### 동적페이지 스크래핑
- 스크래핑하는 웹서버에 부하를 많이 주는 작업
- 웹사이트 오픈, 자동 로그인, 해당페이지로 이동, 필요 내용 검색, 검색결과 스크래핑, 다음 페이지 이동, 검색결과 스크래핑, 반복...

### 주요 사이트 스크래핑
- 다나와 : 난이도 중, (상품 검색, 가격비교, 스펙, ... ), JS로 페이지 변환으로 명시적 대기로 로딩시간 필요, User-Agent 미체크시 접근 거부
- 네이버 : 난이도 상, 포털, CSS클래스명을 시간별로 변경, 연속으로 스크래핑 요청시 캡차 호출
- 구글 : 난이도 상, 포털, 봇 감지, 동적 페이지구조 변경, 연속 스크래핑에 캡차 호출
- 쿠팡 : 난이도 극상, 쇼핑몰, Anti-bot 솔루션 운영, 봇 접근시 403에러 발생, 셀레니움으로 접근 IP차단

### 동적 스크래핑 실습 사이트
- https://quotes.toscrape.com/
- 유명인사들의 명언 등을 보여주는 사이트
- 스크래핑을 위해 개발된 샘플 사이트

In [3]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [4]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://quotes.toscrape.com/')

footer = wait.until(
    EC.presence_of_element_located((By.CLASS_NAME, 'footer'))
)

##### 자동 로그인 연습

- wait.until 쓰는이유 : 다음 페이지 화면이 전환됐을 때 모두 표시되기 전에 스크래핑 시작하면 오류가 발생. 화면이 무조건 다 로드된 이후 작업을 시작하라는 의미 / time.sleep() 보다 효율적

- time.sleep(10) : 10초를 무조건 기다림.

In [8]:
# 로그인 링크 찾아서 클릭
# wait.until 좋은점 화면이 다 뜨고 작업을 하기에 화면이 로드되기 전부터 행동을 하여 오류가 뜨는 것을 방지함
login_link = wait.until(
    EC.element_to_be_clickable((By.LINK_TEXT, 'Login'))
)

login_link.click()

In [11]:
# 아이디 패스워드 입력
username_input = wait.until(
    EC.presence_of_element_located((By.ID, 'username'))
)

password_input = wait.until(
    EC.presence_of_element_located((By.ID, 'password'))
)

username_input.send_keys('admin')
password_input.send_keys('admin')

In [ ]:
# 로그인 버튼 클릭 
login_btn = driver.find_element(By.CSS_SELECTOR, 'input[type="submit"]')
#submit 버튼은  -> 값(data)을 서버에 보내는 역할
#일반적인 버튼 -> 그냥 링크 역할임. (ex. 창이동)

login_btn.click()

In [13]:
# 로그인이 완료된 것 확인
wait.until(
    EC.presence_of_element_located((By.LINK_TEXT, 'Logout'))
)

print('로그인 성공 : 작업 시작')

로그인 성공 : 작업 시작


### 인용문 찾기

In [16]:
quotes = driver.find_elements(By.CLASS_NAME,'quote')

len(quotes)

10

In [33]:
# 첫번째 인용문 확인

quotes = driver.find_elements(By.CLASS_NAME, 'quote')

first = quotes[0]
first.text

text = first.find_element(By.CSS_SELECTOR, '.text').text
author = first.find_element(By.CSS_SELECTOR, '.author').text
link = first.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
tags = first.find_elements(By.CSS_SELECTOR, '.tags > a')
tag_text = ','.join(tag.text for tag in tags)

print(text)
print(author)
print(link)
print(tag_text)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Albert Einstein
http://goodreads.com/author/show/9810.Albert_Einstein
change,deep-thoughts,thinking,world


##### 한 페이지 수집하기

In [ ]:
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

datas = []

for first in quotes:

    text = first.find_element(By.CSS_SELECTOR, '.text').text
    author = first.find_element(By.CSS_SELECTOR, '.author').text
    link = first.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
    tags = first.find_elements(By.CSS_SELECTOR, '.tags > a')
    tag_text = ','.join(tag.text for tag in tags)

    datas.append({
        'text' : text,
        'author' : author,
        'link' : link,
        'tags' : tag_text
    })
    

In [36]:
df = pd.DataFrame(datas)

df

,text,author,link,tags
0,“The world as we have created it is a process ...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"change,deep-thoughts,thinking,world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,http://goodreads.com/author/show/1077326.J_K_R...,"abilities,choices"
2,“There are only two ways to live your life. On...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"inspirational,life,live,miracle,miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,http://goodreads.com/author/show/1265.Jane_Austen,"aliteracy,books,classic,humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,http://goodreads.com/author/show/82952.Marilyn...,"be-yourself,inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"adulthood,success,value"
6,“It is better to be hated for what you are tha...,André Gide,http://goodreads.com/author/show/7617.Andr_Gide,"life,love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,http://goodreads.com/author/show/3091287.Thoma...,"edison,failure,inspirational,paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,http://goodreads.com/author/show/44566.Eleanor...,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,http://goodreads.com/author/show/7103.Steve_Ma...,"humor,obvious,simile"


#### 다음 페이지 이동

In [42]:
# 다음 페이지 정보 확인
next_button = driver.find_element(By.CSS_SELECTOR, 'li.next a')
next_button.click()

#### 여러 페이지 수집하기

In [ ]:
rows = []

driver.get('https://quotes.toscrape.com/')

for page in range(1, 6):
    wait.until(
        EC.presence_of_element_located((By.CLASS_NAME, 'footer'))
    )

    quotes = driver.find_elements(By.CSS_SELECTOR, '.quote')
    for quote in quotes:
        text = quote.find_element(By.CSS_SELECTOR, '.text').text
        author = quote.find_element(By.CSS_SELECTOR, '.author').text
        link = quote.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
        tags = quote.find_elements(By.CSS_SELECTOR, '.tags > a')
        tag_text = ','.join(tag.text for tag in tags)

        rows.append({
            'text' : text,
            'author' : author,
            'link' : link,
            'tags' : tag_text
        })

    next_button = driver.find_element(By.CSS_SELECTOR, 'li.next a')
    if not next_button:
        break
    next_button.click()

In [108]:
len(rows)

50

In [ ]:
# text 컬럼의 중복 개수 확인 (True인 것의 합계)


np.int64(2)

10

In [116]:
df_quotes = pd.DataFrame(rows)
df_quotes

,text,author,link,tags
0,“The world as we have created it is a process ...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,"change,deep-thoughts,thinking,world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling,"abilities,choices"
2,“There are only two ways to live your life. On...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,"inspirational,life,live,miracle,miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,https://quotes.toscrape.com/author/Jane-Austen,"aliteracy,books,classic,humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe,"be-yourself,inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,"adulthood,success,value"
6,“It is better to be hated for what you are tha...,André Gide,https://quotes.toscrape.com/author/Andre-Gide,"life,love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,https://quotes.toscrape.com/author/Thomas-A-Ed...,"edison,failure,inspirational,paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,https://quotes.toscrape.com/author/Eleanor-Roo...,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,https://quotes.toscrape.com/author/Steve-Martin,"humor,obvious,simile"


In [123]:
# text 컬럼의 중복 개수 확인 (True인 것의 합계)
counts = df_quotes['author'].value_counts()
counts[counts > 1] - 1

author
Albert Einstein    7
J.K. Rowling       5
Marilyn Monroe     5
Bob Marley         2
Dr. Seuss          2
Mother Teresa      1
Name: count, dtype: int64

In [124]:
counts = df_quotes['author'].value_counts()
counts[counts > 1]

author
Albert Einstein    8
J.K. Rowling       6
Marilyn Monroe     6
Bob Marley         3
Dr. Seuss          3
Mother Teresa      2
Name: count, dtype: int64

In [110]:
df_quotes = df_quotes.drop_duplicates(subset=['text', 'author'])

df_quotes.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    50 non-null     str  
 1   author  50 non-null     str  
 2   link    50 non-null     str  
 3   tags    50 non-null     str  
dtypes: str(4)
memory usage: 1.7 KB


In [111]:
df_quotes

,text,author,link,tags
0,“The world as we have created it is a process ...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,"change,deep-thoughts,thinking,world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling,"abilities,choices"
2,“There are only two ways to live your life. On...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,"inspirational,life,live,miracle,miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,https://quotes.toscrape.com/author/Jane-Austen,"aliteracy,books,classic,humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe,"be-yourself,inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...,"adulthood,success,value"
6,“It is better to be hated for what you are tha...,André Gide,https://quotes.toscrape.com/author/Andre-Gide,"life,love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,https://quotes.toscrape.com/author/Thomas-A-Ed...,"edison,failure,inspirational,paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,https://quotes.toscrape.com/author/Eleanor-Roo...,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,https://quotes.toscrape.com/author/Steve-Martin,"humor,obvious,simile"


In [ ]:
# CSV 저장
# df_quotes.to_csv('../data/quotes_to_scrap_50.csv', encoding='utf=8')